<a href="https://colab.research.google.com/github/Fatma-Zohair/jebel_ali_thermal_plume/blob/main/scripts/01_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data cleaning script

In [1]:
from google.colab import drive

## Starting to do data cleaning in R

###change runtime to R

In [5]:
#Set up my working directory

setwd("/content/drive/MyDrive/jebel_ali_thermal_plume")
getwd()

[1] "/content/drive/MyDrive/jebel_ali_thermal_plume"

In [6]:
#load libraries
library(readr)
library(dplyr)
library(tidyr)



Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [17]:
#read in raw data

# Print the exact file name(s) inside raw_data
list.files("raw_data")

[1] "wind_omdw_omdb_2022_2025_raw.csv"

In [18]:
#read in raw data
wind_data <- read_csv("raw_data/wind_omdw_omdb_2022_2025_raw.csv")
glimpse(wind_data)



Rows: 92537 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (2): station, metar
dbl  (5): lon, lat, drct, sknt, gust
dttm (1): valid

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 92,537
Columns: 8
$ station <chr> "OMDB", "OMDW", "OMDB", "OMDB", "OMDW", "OMDB", "OMDB", "OMDB"…
$ valid   <dttm> 2022-01-01 00:00:00, 2022-01-01 00:00:00, 2022-01-01 00:13:00…
$ lon     <dbl> 55.3656, 55.1610, 55.3656, 55.3656, 55.1610, 55.3656, 55.3656,…
$ lat     <dbl> 25.2539, 24.8970, 25.2539, 25.2539, 24.8970, 25.2539, 25.2539,…
$ drct    <dbl> 100, 110, 110, 40, 150, 340, NA, 10, 330, 330, 360, 350, 20, 3…
$ sknt    <dbl> 10, 3, 6, 12, 9, 10, 6, 9, 8, 16, 9, 15, 13, 16, 13, 7, 17, 8,…
$ gust    <dbl> 23, NA, NA, 27, NA, 24, NA, NA, NA, NA, NA, 26, NA, NA, NA, NA…
$ metar   <chr> "OMDB 010000Z 10010G23KT 050V160 8000 TSRA BKN045 FEW050CB 21/…


In [24]:
# Summary of each numeric column: min, max, mean, and number of NAs
summary(select(wind_data, drct, sknt, gust))

      drct            sknt             gust      
 Min.   :  0.0   Min.   : 0.000   Min.   :14.00  
 1st Qu.:120.0   1st Qu.: 4.000   1st Qu.:20.50  
 Median :200.0   Median : 7.000   Median :24.50  
 Mean   :201.1   Mean   : 7.422   Mean   :25.71  
 3rd Qu.:300.0   3rd Qu.:10.000   3rd Qu.:30.00  
 Max.   :360.0   Max.   :37.000   Max.   :52.00  
 NAs    :2502                     NAs    :45462  

In [20]:
# Keep only Al Maktoum International (OMDW), the closest station to the Jebel Ali complex (~20 km).
wind_data <- wind_data %>% filter(station == "OMDW")
head(wind_data)

station,valid,lon,lat,drct,sknt,gust,metar
<chr>,<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
OMDW,2022-01-01 00:00:00,55.161,24.897,110,3,NA,OMDW 010000Z 11003KT 090V160 8000 -TSRA FEW045CB 19/17 Q1017 WS ALL RWY TEMPO VRB20G40KT 1500 TSRA SCT030CB
OMDW,2022-01-01 01:00:00,55.161,24.897,150,9,NA,OMDW 010100Z 15009KT 5000 TSRA FEW040CB 20/17 Q1018 WS ALL RWY TEMPO VRB20G40KT 1500 SCT030CB
OMDW,2022-01-01 02:00:00,55.161,24.897,360,9,NA,OMDW 010200Z 36009KT 4000 TSRA BKN014 FEW030CB 18/17 Q1020 WS ALL RWY TEMPO VRB20G40KT 1200
OMDW,2022-01-01 03:00:00,55.161,24.897,360,16,NA,OMDW 010300Z 36016KT 9000 -SHRA FEW060TCU 18/17 Q1021 RETSRA WS ALL RWY TEMPO VRB20G40KT
OMDW,2022-01-01 04:00:00,55.161,24.897,60,17,NA,OMDW 010400Z 06017KT CAVOK 18/17 Q1020 WS ALL RWY NOSIG
OMDW,2022-01-01 05:00:00,55.161,24.897,60,9,NA,OMDW 010500Z 06009KT CAVOK 19/17 Q1022 WS ALL RWY NOSIG


In [22]:
#Remove missing data and wrong data
# -Rows without wind speed cannot be used at all.
# -Wind speed above 40 knots (~20 m/s) is unrealistic for this coastal site.
# -Direction must be between 0 and 360 degrees.

wind_data <- wind_data %>%
  filter(!is.na(sknt)) %>%
  filter(sknt >= 0, sknt <= 40) %>%
  filter(is.na(drct) | (drct >= 0 & drct <= 360))
head(wind_data)

station,valid,lon,lat,drct,sknt,gust,metar
<chr>,<dttm>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
OMDW,2022-01-01 00:00:00,55.161,24.897,110,3,NA,OMDW 010000Z 11003KT 090V160 8000 -TSRA FEW045CB 19/17 Q1017 WS ALL RWY TEMPO VRB20G40KT 1500 TSRA SCT030CB
OMDW,2022-01-01 01:00:00,55.161,24.897,150,9,NA,OMDW 010100Z 15009KT 5000 TSRA FEW040CB 20/17 Q1018 WS ALL RWY TEMPO VRB20G40KT 1500 SCT030CB
OMDW,2022-01-01 02:00:00,55.161,24.897,360,9,NA,OMDW 010200Z 36009KT 4000 TSRA BKN014 FEW030CB 18/17 Q1020 WS ALL RWY TEMPO VRB20G40KT 1200
OMDW,2022-01-01 03:00:00,55.161,24.897,360,16,NA,OMDW 010300Z 36016KT 9000 -SHRA FEW060TCU 18/17 Q1021 RETSRA WS ALL RWY TEMPO VRB20G40KT
OMDW,2022-01-01 04:00:00,55.161,24.897,60,17,NA,OMDW 010400Z 06017KT CAVOK 18/17 Q1020 WS ALL RWY NOSIG
OMDW,2022-01-01 05:00:00,55.161,24.897,60,9,NA,OMDW 010500Z 06009KT CAVOK 19/17 Q1022 WS ALL RWY NOSIG
